# 2.3 밴딧 vs 완전한 MDP — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter02_3_bandit_to_mdp.ipynb)

책 본문: [2.3 밴딧 vs 완전한 MDP: 다음 장으로 가는 다리](https://smhanlab.com/book-ml/kor/ml2/chapter02/3.html)

본문의 세 코드 블록(A·B·C)을 실행하고, D(자주 하는 실수 섹션의
비정상성 예제)를 하나 더 붙였습니다.

1. **A.** 밴딧(= 1-상태 MDP)에서 ε-greedy 실행 — ε=0.1 vs ε=0.0의 '후회' 차이
2. **B.** 1-상태 MDP의 벨만방정식 \(V(s^*) = q^*/(1-\gamma)\) — 반복 대입과 닫힌해
3. **C.** 2-상태 MDP의 벨만방정식 — 반복 대입이 닫힌해 \(V(s_0)=28, V(s_1)=30\)으로 수렴
4. **D.** 비정상성: 보상이 t=1000에서 바뀌면 \(1/N\) 갱신 vs 고정 \(\alpha\) 갱신


## 0. 환경 준비


In [1]:
import numpy as np
import random
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

import os
gamma = 0.9
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)


numpy 2.4.6 | matplotlib 3.11.1


## A. 밴딧(= 1-상태 MDP)에서 ε-greedy 실행

본문 코드 그대로. 1-상태 MDP에서는 정책이 'k개 팔 중 하나를 고르는 한 줄 규칙'
으로 축소되므로, 2.1절의 ε-greedy 코드가 그대로 1-상태 MDP의 정책 평가가 된다.


In [2]:
# 2.1절 코드 그대로. '팔' = 1-상태 MDP의 '행동'.
def run_bandit(true_means, epsilon, steps, seed=0):
    random.seed(seed)
    k = len(true_means)
    Q, N, total = [0.0]*k, [0]*k, 0.0
    for t in range(steps):
        a = random.randrange(k) if random.random() < epsilon \
            else max(range(k), key=lambda i: Q[i])
        r = random.gauss(true_means[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        total += r
    return Q, N, total

true_means = [1.0, 1.5, 2.0]
steps = 2000
Q1, N1, tot1 = run_bandit(true_means, epsilon=0.1, steps=steps)
Q0, N0, tot0 = run_bandit(true_means, epsilon=0.0, steps=steps)
opt = steps * max(true_means)   # '매번 진짜 최선의 팔만 골랐다면'의 총 보상
print("ε=0.1  Q =", [round(x, 3) for x in Q1], " N =", N1, " 총보상 =", round(tot1, 1),
      " 후회 =", round(opt - tot1, 1))
print("ε=0.0  Q =", [round(x, 3) for x in Q0], " N =", N0, " 총보상 =", round(tot0, 1),
      " 후회 =", round(opt - tot0, 1))


ε=0.1  Q = [0.978, 1.315, 2.006]  N = [82, 50, 1868]  총보상 = 3893.9  후회 = 106.1
ε=0.0  Q = [0.977, 0.0, 0.0]  N = [2000, 0, 0]  총보상 = 1953.7  후회 = 2046.3


같은 시드(0)에서 \(\varepsilon=0.1\)만 달리면 '후회'가 확 벌어진다 — 
\(\varepsilon=0\)은 첫 팔의 보상이 잡음으로 낮게 나온 **단 한 번의 불운**으로
나머지 1999번을 최악의 팔만 당길 수 있다. 2.1절의 '탐험 없이 활용만은 위험하다'가
숫자로 새겨지는 부분.


## B. 1-상태 MDP의 벨만방정식: \(V(s^*) = q^* + \gamma V(s^*)\)

자기 전이뿐인 1-상태 MDP에서 벨만방정식은 \(V = q^* + \gamma V\)로 축소되고,
닫힌해는 \(V = q^*/(1-\gamma)\). 반복 대입(Chapter 4 정책평가의 1-상태판)과
닫힌해가 일치하는지 \(\gamma\)를 바꿔가며 확인한다.


In [3]:
# V = q* + γ·V 를 반복 대입으로 풀기 (Chapter 4의 정책평가, 1-상태판)
def value_1state(q_star, gamma, iters=500):
    V = 0.0
    for _ in range(iters):
        V = q_star + gamma * V
    return V

q_star = 2.0
for g in [0.0, 0.5, 0.9, 0.99]:
    closed = q_star/(1-g) if g < 1 else float('inf')
    print(f"γ={g}:  V(s*) = {value_1state(q_star, g):.4f}   (닫힌해 {closed:.4f})")


γ=0.0:  V(s*) = 2.0000   (닫힌해 2.0000)
γ=0.5:  V(s*) = 4.0000   (닫힌해 4.0000)
γ=0.9:  V(s*) = 20.0000   (닫힌해 20.0000)
γ=0.99:  V(s*) = 198.6859   (닫힌해 200.0000)


γ=0에서는 '한 스텝의 평균 보상' \(q^*\) 그대로, γ=0.9에서는 \(q^*/(1-\gamma)=20\).
같은 팔·같은 정책인데 'γ가 보는 눈의 깊이'가 다른 두 값.


## C. 2-상태 MDP에서 벨만방정식 직접 풀기

본문의 장난감 MDP: \(s_0\)은 보상 1.0을 받고 \(s_1\)로 전이, \(s_1\)은 보상 3.0을
받고 자기 전이(영원히 \(s_1\)에 남는다). 벨만방정식:

\(V(s_1) = 3.0 + \gamma V(s_1)\)  →  \(V(s_1) = 3.0/(1-\gamma)\)
\(V(s_0) = 1.0 + \gamma V(s_1)\)

\(V_0 = 0\)에서 시작해 반복 대입하면, 이 두 수치가 닫힌해로 기하급수적으로
수렴한다 — Chapter 4의 `policy_evaluation`이 하는 일의 2-상태판이다.


In [4]:
# s0: 보상 1.0, s1로 전이(확률 1).  s1: 보상 3.0, 자기 전이(확률 1).
def two_state_value(gamma=0.9, iters=500):
    V = [0.0, 0.0]   # V[0] = V(s0), V[1] = V(s1)
    for _ in range(iters):
        V[1] = 3.0 + gamma * V[1]   # s1 자기 전이
        V[0] = 1.0 + gamma * V[1]   # s0 -> s1
    return V

V = two_state_value(gamma=0.9)
print(f"γ=0.9:  V(s0) = {V[0]:.4f},  V(s1) = {V[1]:.4f}")
print(f"닫힌해: V(s1) = 3.0/(1-0.9) = {3.0/0.1:.4f},  V(s0) = 1.0 + 0.9*30 = {1.0 + 0.9*30.0:.4f}")


γ=0.9:  V(s0) = 28.0000,  V(s1) = 30.0000
닫힌해: V(s1) = 3.0/(1-0.9) = 30.0000,  V(s0) = 1.0 + 0.9*30 = 28.0000


In [5]:
# 반복 대입의 수렴 과정 — 매 스텝의 (V(s0), V(s1))을 녹화
def two_state_value_trace(gamma=0.9, iters=60):
    V = [0.0, 0.0]
    trace = [[0.0, 0.0]]
    for _ in range(iters):
        V[1] = 3.0 + gamma * V[1]
        V[0] = 1.0 + gamma * V[1]
        trace.append([V[0], V[1]])
    return trace

trace = two_state_value_trace(gamma=0.9, iters=60)
iters = range(len(trace))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(iters, [t[0] for t in trace], "o-", ms=4, label=r"$V(s_0)$ → 28.0")
ax.plot(iters, [t[1] for t in trace], "s-", ms=4, label=r"$V(s_1)$ → 30.0")
ax.axhline(28.0, ls="--", c="gray", lw=1)
ax.axhline(30.0, ls="--", c="gray", lw=1)
ax.set_xlabel("반복 대입 스텝 k")
ax.set_ylabel(r"$V$")
ax.set_title(r"2-상태 MDP 반복 대입의 수렴  ($\gamma=0.9$)")
ax.legend()
fig.tight_layout()
svg = os.path.join(IMG, "ch02_3_two_state_convergence.svg")
fig.savefig(svg)
plt.show()
print(f"그림 저장: {svg}")


그림 저장: /home/smhan/book-ml/kor/src/images/ch02_3_two_state_convergence.svg


V(s1)=30.0, V(s0)=28.0 — 본문에서 손으로 계산한 값과 정확히 일치. 
**상태 간 참조**: V(s0) = 1.0 + 0.9·V(s1) = 1.0 + 27.0 이라, s0의 가치는 s1의
가치에 달려 있다. 1-상태(밴딧)에서는 \(V = q^*/(1-\gamma)\)로 자기 자신만으로
닫혔지만, 2개만 돼도 **연립 방정식**이 되는 순간 — Chapter 4가 이 연립을
일반 \(|S|\)개 상태로 확장해 푸는 것이다.


## D. 비정상성: '한 번 바뀌는' 환경에서 \(1/N\) 갱신 vs 고정 \(\alpha\) 갱신

본문 '자주 하는 실수' 섹션의 예제 — 팔 B의 진짜 평균 보상이 **t=1000에서 1.0 → 3.0**으로
바뀌는 환경. 전이 구조는 여전히 자기 전이뿐(= 밴딧)이지만, 보상 분포가 시간에 바뀐다.

- **\(1/N_t(a)\) 갱신**(2.1 기본 코드): 모든 과거 보상의 평균을 추적 → 전환 후에도 천천히 따라옴
- **고정 \(\alpha\) 갱신**(2.1 FAQ): 최근 관찰에 큰 가중치 → 전환 직후 빠르게 따라옴


In [6]:
# 팔 B(인덱스 1)의 진짜 평균이 t=1000에서 1.0 -> 3.0으로 한 번 바뀐다.
def run_nonstationary(epsilon, alpha, steps=2000, switch=1000, seed=0):
    random.seed(seed)
    k = 3
    means = [1.5, 1.0, 2.0]
    Q, N = [0.0]*k, [0]*k
    for t in range(steps):
        if t >= switch:
            means[1] = 3.0
        a = random.randrange(k) if random.random() < epsilon \
            else max(range(k), key=lambda i: Q[i])
        r = random.gauss(means[a], 1.0)
        N[a] += 1
        if alpha is None:        # 1/N 갱신 (2.1 기본 코드)
            Q[a] += (r - Q[a]) / N[a]
        else:                    # 고정 α 갱신 (2.1 FAQ)
            Q[a] += alpha * (r - Q[a])
    return Q, N

Q_1N, N_1N = run_nonstationary(epsilon=0.1, alpha=None)
Q_alpha, N_alpha = run_nonstationary(epsilon=0.1, alpha=0.2)
print("전환 후(2000스텝) 팔 B 추정치:")
print(f"  1/N 갱신:   Q[B] = {Q_1N[1]:.3f}  (진짜 평균 3.0 — 전환점 이후로는 천천히 따라옴)")
print(f"  α=0.2 갱신: Q[B] = {Q_alpha[1]:.3f}  (진짜 평균 3.0 — 전환 직후 빠르게 접근)")
print("(참고: A, C의 추정치는 두 정책 모두 대략 정확한 편)")
print(f"  1/N:      Q = {[round(x,2) for x in Q_1N]}")
print(f"  α=0.2:    Q = {[round(x,2) for x in Q_alpha]}")


전환 후(2000스텝) 팔 B 추정치:
  1/N 갱신:   Q[B] = 1.735  (진짜 평균 3.0 — 전환점 이후로는 천천히 따라옴)
  α=0.2 갱신: Q[B] = 3.394  (진짜 평균 3.0 — 전환 직후 빠르게 접근)
(참고: A, C의 추정치는 두 정책 모두 대략 정확한 편)
  1/N:      Q = [1.5, 1.74, 2.01]
  α=0.2:    Q = [1.47, 3.39, 2.07]


## 마무리: 이 세(네) 가지를 한 줄로

- **A**는 '밴딧에서도 탐험이 필요하다' — ε=0의 한 번의 불운이 2000스텝을 망친다
- **B**는 '밴딧의 \(q^*(a)\)가 MDP의 \(V^\pi(s)\)의 \(\gamma=0\) 극한' — \(V = q^*/(1-\gamma)\)
- **C**는 '상태가 2개만 돼도 가치함수가 연립 방정식이 된다' — \(V(s_0)=28, V(s_1)=30\)
- **D**는 '상태가 하나' ≠ '보상이 고정' — 전이(구조)와 비정상성(분포)은 다른 층의 문제

다음: [3.3 가치함수와 벨만 기대방정식 실습](chapter03_3_bellman_value.ipynb) — 같은 벨만방정식을
일반 \(|S|\)개 상태로 확장해 `policy_evaluation` 함수로 푸는 곳.
